In [3]:
import re
import pandas as pd
import requests

In [4]:
file_path = "./input/rebase/link_withref.txt"

In [5]:

def download_rebase_file(file_path, url="https://rebase.neb.com/rebase/link_withref"):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(file_path, "w") as fh:
            fh.write(response.text)
        print("Downloaded to", file_path)
    else:
        print("Failed to download file. Status code:", response.status_code)


In [6]:
download_rebase_file(file_path)

Downloaded to ./input/rebase/link_withref.txt


In [7]:
# Helper function to extract methylation types
def extract_methylation_type_number(methylation_str):
    return list(map(int, re.findall(r'\((\d+)\)', methylation_str))) if methylation_str else []

# Helper function to annotate methylation types
def annotate_methylation_types(types):
    annotations = []
    for t in types:
        if t == 6:
            annotations.append("N6-methyladenine")
        elif t == 5:
            annotations.append("5-methylcytosine")
        elif t == 4:
            annotations.append("N4-methylcytosine")
        elif t == 99:
            annotations.append("unknown methylation site")
        else:
            annotations.append(f"unrecognized ({t})")
    return ", ".join(annotations) if annotations else ""

def filter_methylation(methylation_types):
    """
    Returns False if methylation_types contains 5, 6, or 99; otherwise, returns True.
    
    Parameters:
      methylation_types (list): List of methylation type numbers.
      
    Returns:
      bool: False if any number in the list is 5, 6, or 99, True otherwise.
    """
    if any(m in [5, 6, 99] for m in methylation_types):
        return True
    return False

In [8]:
def parse_rebase_file_to_df_with_annotations(file_path):
    """
    Parse a Rebase file and annotate methylation types.

    Reads the file at the specified file_path containing enzyme entries formatted with
    numbered tags (e.g., <1>, <2>, etc.). Each enzyme entry is separated by the "<1>" tag.
    For each entry, the function extracts key enzyme information including name, prototype,
    organism, source, recognition sequence, methylation details, and supplier. The methylation
    string is processed by extracting numbers using the helper function `extract_methylation_type_number`
    and by annotating these types with `annotate_methylation_types`.

    Parameters:
        file_path (str): The path to the input file that contains the enzyme data.

    Returns:
        pd.DataFrame: A DataFrame containing the parsed and annotated enzyme information.
                    Columns include:
                      - enzyme: Enzyme name
                      - prototype: Enzyme prototype
                      - organism: Source organism
                      - source: Data source
                      - recognition_seq: Recognition sequence
                      - methylation: Raw methylation string from the file
                      - methylation_types: Extracted list of methylation type numbers
                      - methylation_note: Annotated methylation descriptions
                      - supplier: Supplier information
                      - reference: Reference field
    """
    # Read the file
    with open(file_path, 'r') as f:
        content = f.read()

    # Split into individual enzyme entries based on <1> tag
    entries = content.split("<1>")[1:]

    # Prepare data list
    data = []

    for entry in entries:
        entry = "<1>"+entry.strip()
        lines = entry.split("\n")
        entry_dict = {}
        for line in lines:
            match = re.match(r"<(\d+)>(.*)", line)
            if match:
                key, value = match.groups()
                entry_dict[int(key)] = value.strip()

        data.append({
            "enzyme": entry_dict.get(1, ""),
            "prototype": entry_dict.get(2, ""),
            "organism": entry_dict.get(3, ""),
            "source": entry_dict.get(4, ""),
            "recognition_seq": entry_dict.get(5, ""),
            "methylation": entry_dict.get(6, ""),
            "supplier": entry_dict.get(7, ""),
            "reference": entry_dict.get(8, ""),
        })

    return pd.DataFrame(data)

In [9]:
df = parse_rebase_file_to_df_with_annotations(file_path)
df['methylation_types'] = df['methylation'].apply(extract_methylation_type_number)
df['methylation_note'] = df['methylation_types'].apply(annotate_methylation_types)
df['6mA_5mC_sensitive'] = df['methylation_types'].apply(filter_methylation)

In [10]:
df

,enzyme,prototype,organism,source,recognition_seq,methylation,supplier,reference,methylation_types,methylation_note,6mA_5mC_sensitive
0,AaaI,XmaIII,Acetobacter aceti ss aceti,M. Fukaya,C^GGCCG,,,"Tagami, H., Tayama, K., Tohyama, T., Fukaya, M...",[],,False
1,I-AabMI,,Ascocalyx abietina,B.L. Stoddard,CAGGTACCCTTTAAACCTACTAACCC(-12/-16),,,"Lambert, A.R., Hallinan, J.P., Shen, B.W., Chi...",[],,False
2,AacLI,BamHI,Acetobacter aceti sub. liquefaciens,IFO 12388,GGATCC,,,"Seurinck, J., van Montagu, M., Unpublished obs...",[],,False
3,AaeI,BamHI,Acetobacter aceti sub. liquefaciens,M. Van Montagu,GGATCC,,,"Seurinck, J., van Montagu, M., Unpublished obs...",[],,False
4,AagI,ClaI,Achromobacter agile,N.N. Sokolov,AT^CGAT,,,"Sokolov, N.N., Maneliene, Z.P., Butkus, V.V., ...",[],,False
...,...,...,...,...,...,...,...,...,...,...,...
6059,M.ZmoIII,HinfI,Zymomonas mobilis subsp. mobilis ZM4,ATCC 31821,GANTC,2(6),,"Anton, B., Unpublished observations.",[6],N6-methyladenine,True
6060,M.Zmo29192III,HinfI,Zymomonas mobilis subsp. pomaceae ATCC 29192,ATCC 29192,GANTC,2(6),,"Blow, M.J. et al., (2016) PLoS Genet., vol. 12.",[6],N6-methyladenine,True
6061,ZraI,AatII,Zoogloea ramigera 11,NEB 1785,GAC^GTC,,INV,"Dedkov, V.S., Sinichkina, S.A., Popichenko, D....",[],,False
6062,ZrmI,ScaI,Zoogloea ramigera SCA,S.K. Degtyarev,AGT^ACT,,IV,"Nadeev, A.N., Kileva, E.V., Popichenko, D.V., ...",[],,False


In [11]:
df.to_csv("./output/methylation_check.csv", index=False)

In [12]:
df = pd.read_csv("./output/methylation_check.csv")

In [13]:
df

,enzyme,prototype,organism,source,recognition_seq,methylation,supplier,reference,methylation_types,methylation_note,6mA_5mC_sensitive
0,AaaI,XmaIII,Acetobacter aceti ss aceti,M. Fukaya,C^GGCCG,NaN,NaN,"Tagami, H., Tayama, K., Tohyama, T., Fukaya, M...",[],NaN,False
1,I-AabMI,NaN,Ascocalyx abietina,B.L. Stoddard,CAGGTACCCTTTAAACCTACTAACCC(-12/-16),NaN,NaN,"Lambert, A.R., Hallinan, J.P., Shen, B.W., Chi...",[],NaN,False
2,AacLI,BamHI,Acetobacter aceti sub. liquefaciens,IFO 12388,GGATCC,NaN,NaN,"Seurinck, J., van Montagu, M., Unpublished obs...",[],NaN,False
3,AaeI,BamHI,Acetobacter aceti sub. liquefaciens,M. Van Montagu,GGATCC,NaN,NaN,"Seurinck, J., van Montagu, M., Unpublished obs...",[],NaN,False
4,AagI,ClaI,Achromobacter agile,N.N. Sokolov,AT^CGAT,NaN,NaN,"Sokolov, N.N., Maneliene, Z.P., Butkus, V.V., ...",[],NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
6059,M.ZmoIII,HinfI,Zymomonas mobilis subsp. mobilis ZM4,ATCC 31821,GANTC,2(6),NaN,"Anton, B., Unpublished observations.",[6],N6-methyladenine,True
6060,M.Zmo29192III,HinfI,Zymomonas mobilis subsp. pomaceae ATCC 29192,ATCC 29192,GANTC,2(6),NaN,"Blow, M.J. et al., (2016) PLoS Genet., vol. 12.",[6],N6-methyladenine,True
6061,ZraI,AatII,Zoogloea ramigera 11,NEB 1785,GAC^GTC,NaN,INV,"Dedkov, V.S., Sinichkina, S.A., Popichenko, D....",[],NaN,False
6062,ZrmI,ScaI,Zoogloea ramigera SCA,S.K. Degtyarev,AGT^ACT,NaN,IV,"Nadeev, A.N., Kileva, E.V., Popichenko, D.V., ...",[],NaN,False
